# Title1
## Title2
**bold**

*italic*
* bulletpoint
  

In [1]:
#This is intended to synchronize src files. We don't need in case we don't have src files included

%load_ext autoreload
%autoreload 2


In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

In [3]:
df = pd.read_excel("/Users/Conny/Documents/340ks/SCREEN-Analysis/PO-ini2810-3.xlsx")
df.shape    # only for control while programming, shows number of (rows, columns)

(176000, 17)

In [4]:
df

,ID,Cxsmiles,Formula,MW,Purity,PO,Salt smiles,Salt_name,Salt_ratio,MW_salt,Plate_ID,Well,"V, nL","Conc, mM",Chemical name,Thermal_Shift,Spectral_Shift
0,BH4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1969721-001,A01,40,10,NaN,49.2134,2.656585
1,BH4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1969721-001,A02,40,10,NaN,49.3938,2.636959
2,CSCS00104537142,CC(C(=O)N1CCC(C1)(C(=O)O)C(F)(F)F)N2C=NC=N2,C11H13F3N4O3,306.2411,100.0,ini2810,NaN,NaN,NaN,0.0000,1969721-001,A03,40,10,"1-[2-(1H-1,2,4-triazol-1-yl)propanoyl]-3-(trif...",49.4840,2.963650
3,CSCS00133135693,CC1=NOC(=N1)C2CCN(CC2)C(=O)NC3=CC=C(F)C(F)=C3F,C15H15F3N4O2,340.3004,92.5,ini2810,NaN,NaN,NaN,0.0000,1969721-001,A04,40,10,"4-(3-methyl-1,2,4-oxadiazol-5-yl)-N-(2,3,4-tri...",49.4840,2.951655
4,CSCS00093835758,CN(CC(=O)NC1CCS(=O)(=O)C1)C=2N=CN=C3C=CSC23,C13H16N4O3S2,340.4211,100.0,ini2810,NaN,NaN,NaN,0.0000,1969721-001,A05,40,10,"N-(1,1-dioxo-1lambda6-thiolan-3-yl)-2-[methyl(...",49.6643,2.948427
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175995,CSCS00133030229,CC(C)(C=1N=NN(CCN2C=C(Cl)C=N2)N1)S(C)(=O)=O,C10H15ClN6O2S,318.7831,100.0,ini2810,NaN,NaN,NaN,0.0000,C1969721-250,P18,40,10,2-[2-(4-chloro-1H-pyrazol-1-yl)ethyl]-5-(2-met...,NaN,NaN
175996,CSCS00027831295,CC1CN(CC(O1)C2CC2)C3=NC=NN3C,C11H18N4O,222.2868,99.0,ini2810,NaN,NaN,NaN,0.0000,C1969721-250,P19,40,10,"2-cyclopropyl-6-methyl-4-(1-methyl-1H-1,2,4-tr...",NaN,NaN
175997,CSCS00133127202,CC=1N=C(SC1C(=O)O)C(C)NC(=O)C=2C=CC=CC2C3CCC3,C18H20N2O3S,344.4280,100.0,ini2810,NaN,NaN,NaN,0.0000,C1969721-250,P20,40,10,2-{1-[(2-cyclobutylphenyl)formamido]ethyl}-4-m...,NaN,NaN
175998,CSCS00025925683,CC(NC(=O)NCC(C)(C)N1CCOCC1)C2=NN=CN2C,C14H26N6O2,310.3952,91.0,ini2810,NaN,NaN,NaN,0.0000,C1969721-250,P21,40,10,1-[2-methyl-2-(morpholin-4-yl)propyl]-3-[1-(4-...,NaN,NaN


In [5]:
ID = 'DMSO'
df_DMSO = df.loc[df['ID'] == ID]
# df_DMSO

In [6]:
uniq_Plate_IDs = df_DMSO['Plate_ID'].unique()

In [12]:
columns_table = ['Plate_ID', 'Protein', 'Compound'] + [f'Rep{i}' for i in range(1, 17)] + ['Median', 'IQR', 'IQR_rel[%]', 'Median_filtered', 'Mean', 'STD', 'Mean_filtered', 'STD_filtered']
TSA_DMSO_Controls_PAH = pd.DataFrame(columns = columns_table)    #creates empty table with specified columns

#Loop for TSA: DMSO negative control distribution analysis

for i in tqdm(range(len(uniq_Plate_IDs))):
    plate_ID = uniq_Plate_IDs[i]
    protein = 'X'                     ### THIS MIGHT BE WRONG
    compound = 'DMSO'
    
    df_temp = df.loc[(df['ID'] == 'DMSO') & (df['Plate_ID'] == uniq_Plate_IDs[i])]    # creates a subset of df, where additionally only one Plate ID is shown  ()&()
    df_temp = df_temp.dropna(subset = ['Thermal_Shift'])                              # discards all lines where Thermal Shift is NaN
    #df_temp
    
    Tm = df_temp['Thermal_Shift']     # creates a one column table
    Tm = np.array(Tm)                  # converts one col table to list

    # Non-parametric method
    x = Tm

    if len(x) == 0:
        # print(f"No Tm values for Plate_ID {plate_ID} – skip.")
        continue

    median = np.median(x)

    Q1 = np.percentile(x, 25)
    Q3 = np.percentile(x, 75)
    IQR = Q3 - Q1
    IQR_rel = IQR/median*100
   # IQR_filtered = 1000                ### BEHNAM PLEASE HELP

    x_filtered = x[(x >= (Q1 - 1.5*IQR)) &  (x <= (Q3 + 1.5*IQR))]    # filter x values outside [Q1 - 1.5*IQR , Q3 + 1.5*IQ]
    median_filtered = np.median(x_filtered)                           # calculate median again without outliers (however, median is already robust)

   # Parametric method       # I CHANGED X TO Y HERE
    y = Tm

    mean = np.mean(y)
    sd = np.std(y)

    thr = 3               # 3 sigma
    z = (y - mean) / sd

    y_filtered = y[(z >= -thr) &  (z <= thr)]

    mean_filtered = np.mean(y_filtered)
    sd_filtered = np.std(y_filtered)

    Tm = Tm.tolist()      # convert np.array correctly to list (alternative: Tm = list(Tm) gives the np.float64(xyz) ..)

    newline = [plate_ID, protein, compound] + Tm + [median.item(), IQR.item(), IQR_rel.item(), median_filtered.item(), mean.item(), sd.item(), mean_filtered.item(), sd_filtered.item()]             

    TSA_DMSO_Controls_PAH.loc[len(TSA_DMSO_Controls_PAH)] = newline      # len(df) is number of existing lines; here, "newline" is added at the end of the table

100%|█████████████████████████████████████████| 500/500 [00:08<00:00, 61.35it/s]


In [13]:
#columns_table = ['Plate_ID', 'Protein', 'Compound'] + [f'Rep{i}' for i in range(1, 17)] + ['Median', 'IQR', 'IQR_rel [%]', 'Median_filtered', 'Mean', 'STD', 'Mean_filtered', 'STD_filtered']#columns_table = ['Plate_ID', 'Protein', 'Compound'] + [f'Rep{i}' for i in range(1, 17)] + ['Median', 'IQR', 'IQR_rel [%]', 'Median_filtered', 'Mean', 'STD', 'Mean_filtered', 'STD_filtered']
SpSh_DMSO_Controls_PAH = pd.DataFrame(columns = columns_table)    #creates empty table with specified columns

#Loop for Spectral Shift: DMSO negative control distribution analysis

for i in tqdm(range(len(uniq_Plate_IDs))):
    plate_ID = uniq_Plate_IDs[i]
    protein = 'X'                     ### THIS MIGHT BE WRONG
    compound = 'DMSO'
    
    df_temp = df.loc[(df['ID'] == 'DMSO') & (df['Plate_ID'] == uniq_Plate_IDs[i])]    # creates a subset of df, where additionally only one Plate ID is shown  ()&()
    df_temp = df_temp.dropna(subset = ['Spectral_Shift'])                              # discards all lines where Thermal Shift is NaN
    #df_temp

    SpSh = df_temp['Spectral_Shift']     # creates a one column table
    SpSh = np.array(SpSh)                  # converts one col table to list

    # Non-parametric method
    x = SpSh

    if len(x) == 0:
        # print(f"No SpSh values for Plate_ID {plate_ID} – skip.")
        continue

    median = np.median(x)

    Q1 = np.percentile(x, 25)
    Q3 = np.percentile(x, 75)
    IQR = Q3 - Q1
    IQR_rel = IQR/median*100

    x_filtered = x[(x >= (Q1 - 1.5*IQR)) &  (x <= (Q3 + 1.5*IQR))]    # filter x values outside [Q1 - 1.5*IQR , Q3 + 1.5*IQ]
    median_filtered = np.median(x_filtered)                           # calculate median again without outliers (however, median is already robust)

   # Parametric method       # I CHANGED X TO Y HERE
    y = SpSh

    mean = np.mean(y)
    sd = np.std(y)

    thr = 3               # 3 sigma
    z = (y - mean) / sd

    y_filtered = y[(z >= -thr) &  (z <= thr)]

    mean_filtered = np.mean(y_filtered)
    sd_filtered = np.std(y_filtered)

    SpSh = SpSh.tolist()      # convert np.array correctly to list (alternative: SpSh = list(SpSh) gives the np.float64(xyz) ..)

    newline = [plate_ID, protein, compound] + SpSh + [median.item(), IQR.item(), IQR_rel.item(), median_filtered.item(), mean.item(), sd.item(), mean_filtered.item(), sd_filtered.item()]             

    SpSh_DMSO_Controls_PAH.loc[len(SpSh_DMSO_Controls_PAH)] = newline      # len(df) is number of existing lines; here, "newline" is added at the end of the table

100%|█████████████████████████████████████████| 500/500 [00:08<00:00, 61.11it/s]


In [9]:
TSA_DMSO_Controls_PAH

,Plate_ID,Protein,Compound,Rep1,Rep2,Rep3,Rep4,Rep5,Rep6,Rep7,...,Rep15,Rep16,Median,IQR,IQR_rel [%],Median_filtered,Mean,STD,Mean_filtered,STD_filtered
0,1969721-001,X,DMSO,49.4840,49.3938,49.4840,49.4840,50.0251,49.3938,49.4840,...,49.4840,49.4840,49.4840,0.000000,0.000000,49.4840,49.495263,0.155799,49.459940,0.076998
1,1969721-002,X,DMSO,49.7545,49.2134,49.2134,49.3938,48.9429,49.3938,49.3938,...,49.7545,49.3036,49.3487,0.180400,0.365562,49.3036,49.343056,0.193861,49.343056,0.193861
2,1969721-003,X,DMSO,49.2134,49.0331,49.4840,49.2134,49.3036,49.2134,49.1232,...,49.3938,49.0331,49.2134,0.180400,0.366567,49.2134,49.202144,0.134802,49.202144,0.134802
3,1969721-004,X,DMSO,49.3036,49.7545,49.4840,49.7545,49.3036,49.2134,49.5741,...,49.4840,49.4840,49.4840,0.135200,0.273220,49.4840,49.489606,0.151129,49.489606,0.151129
4,1969721-005,X,DMSO,49.4840,49.1232,49.3938,49.1232,49.3036,49.3036,49.1232,...,49.2134,49.0331,49.3036,0.202950,0.411633,49.3036,49.275419,0.130384,49.275419,0.130384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318,C1969721-213,X,DMSO,49.3036,48.9429,49.2134,49.1232,49.3938,49.0331,49.0331,...,49.0331,48.9429,49.1232,0.202850,0.412941,49.1232,49.134519,0.149113,49.134519,0.149113
319,C1969721-214,X,DMSO,49.1232,49.1232,48.9429,48.9429,48.8527,48.8527,49.3036,...,49.1232,48.9429,48.9429,0.135275,0.276394,48.9429,48.993612,0.123347,48.993612,0.123347
320,C1969721-215,X,DMSO,49.2134,49.3036,48.7625,49.1232,49.2134,49.3938,49.3938,...,49.4840,49.0331,49.2134,0.202850,0.412184,49.2134,49.173969,0.177432,49.173969,0.177432
321,C1969721-216,X,DMSO,49.4840,49.4840,49.3938,49.1232,49.3938,49.1232,49.4840,...,49.4840,49.3938,49.3938,0.293150,0.593496,49.3938,49.348706,0.159440,49.348706,0.159440


In [10]:
SpSh_DMSO_Controls_PAH

,Plate_ID,Protein,Compound,Rep1,Rep2,Rep3,Rep4,Rep5,Rep6,Rep7,...,Rep15,Rep16,Median,IQR,IQR_rel [%],Median_filtered,Mean,STD,Mean_filtered,STD_filtered
0,1969721-001,X,DMSO,2.881563,2.897639,2.905182,2.920294,2.894495,2.932496,2.890735,...,2.955143,2.871904,2.903465,0.031402,1.081550,2.903465,2.910569,0.026390,2.910569,0.026390
1,1969721-002,X,DMSO,2.951799,3.002978,2.988714,3.003054,2.964993,3.051778,3.014130,...,2.986986,3.028467,3.001054,0.034008,1.133189,3.001054,2.996566,0.029113,2.996566,0.029113
2,1969721-003,X,DMSO,2.914656,2.951718,2.993987,3.021784,2.990126,2.964272,2.924410,...,2.979026,3.000899,2.976828,0.049121,1.650105,2.976828,2.973198,0.033037,2.973198,0.033037
3,1969721-004,X,DMSO,2.994405,2.982801,2.975357,2.997645,2.999373,2.950948,2.941710,...,2.951623,2.960281,2.972701,0.043583,1.466115,2.972701,2.978490,0.026960,2.978490,0.026960
4,1969721-005,X,DMSO,2.886937,2.941577,2.850554,2.909828,2.915589,2.914835,2.866566,...,2.848845,2.916313,2.897231,0.044305,1.529220,2.897231,2.891629,0.030287,2.891629,0.030287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,C1969721-213,X,DMSO,2.989165,2.987066,2.988256,2.993577,2.984239,2.969020,2.970776,...,2.955835,2.999956,2.984448,0.022759,0.762601,2.984448,2.976813,0.017307,2.976813,0.017307
339,C1969721-214,X,DMSO,2.997737,3.031316,3.016785,2.996505,3.018450,2.970322,2.983102,...,3.027625,2.995768,2.994834,0.028065,0.937116,2.995768,2.987984,0.032323,2.987984,0.032323
340,C1969721-215,X,DMSO,2.881406,2.910658,3.010305,2.914597,2.949888,2.923059,2.907757,...,2.895533,2.896184,2.909207,0.034115,1.172642,2.907757,2.918589,0.033365,2.918589,0.033365
341,C1969721-216,X,DMSO,2.982045,3.018421,3.028959,3.073579,3.007634,2.996755,3.000088,...,2.998761,2.987816,2.992286,0.030283,1.012031,2.992286,2.993627,0.035417,2.993627,0.035417


In [38]:
TSA_DMSO_Controls_PAH.to_csv('TSA_DMSO_Controls_PAH.csv', index=False)
SpSh_DMSO_Controls_PAH.to_csv('SpSh_DMSO_Controls_PAH.csv', index=False)

In [14]:
# adds delta_Tm values and DMSO-warnings

df['delta_Tm_median'] = None
df['delta_Tm_mean'] = None
df['Tm_IQR_DMSO_Warning'] = None
df['Tm_abs_DMSO_Warning'] = None

for i in tqdm(range(len(df))):
    Thermal_Shift = df.loc[i, 'Thermal_Shift']
    Plate_ID = df.loc[i, 'Plate_ID']

    Median_DMSO = TSA_DMSO_Controls_PAH.loc[TSA_DMSO_Controls_PAH['Plate_ID'] == Plate_ID, 'Median']
    Mean_DMSO = TSA_DMSO_Controls_PAH.loc[TSA_DMSO_Controls_PAH['Plate_ID'] == Plate_ID, 'Mean']
    IQR_rel_DMSOwarn = TSA_DMSO_Controls_PAH.loc[TSA_DMSO_Controls_PAH['Plate_ID'] == Plate_ID, 'IQR_rel[%]']
    
    if Median_DMSO.empty or Mean_DMSO.empty:
        continue
        
    deltaTm_median = Thermal_Shift - Median_DMSO
    deltaTm_mean = Thermal_Shift - Mean_DMSO

    df.loc[i, 'delta_Tm_median'] = deltaTm_median.item()
    df.loc[i, 'delta_Tm_mean'] = deltaTm_mean.item()    

    if IQR_rel_DMSOwarn.item() >1:
        df.loc[i, 'Tm_IQR_DMSO_Warning'] = 'Warning: high IQR for DMSO controls!'

    if Median_DMSO.item() <48.7:                                                     ######### THIS VALUE IS ONLY TRUE FOR PAH WT !!!!!    
        df.loc[i, 'Tm_abs_DMSO_Warning'] = 'Warning: low Tm for DMSO controls!'
        

100%|█████████████████████████████████| 176000/176000 [01:27<00:00, 2018.70it/s]


In [15]:
# adds delta_SpSh values and DMSO-warnings

df['delta_SpSh_median'] = None
df['delta_SpSh_mean'] = None
df['SpSh_IQR_DMSO_Warning'] = None
df['SpSh_abs_DMSO_Warning'] = None

for i in tqdm(range(len(df))):
    Spectral_Shift = df.loc[i, 'Spectral_Shift']
    Plate_ID = df.loc[i, 'Plate_ID']

    Median_DMSO = SpSh_DMSO_Controls_PAH.loc[SpSh_DMSO_Controls_PAH['Plate_ID'] == Plate_ID, 'Median']
    Mean_DMSO = SpSh_DMSO_Controls_PAH.loc[SpSh_DMSO_Controls_PAH['Plate_ID'] == Plate_ID, 'Mean']
    IQR_rel_DMSOwarn = SpSh_DMSO_Controls_PAH.loc[SpSh_DMSO_Controls_PAH['Plate_ID'] == Plate_ID, 'IQR_rel[%]']
    
    if Median_DMSO.empty or Mean_DMSO.empty:
        continue
        
    deltaSpSh_median = Spectral_Shift - Median_DMSO
    deltaSpSh_mean = Spectral_Shift - Mean_DMSO

    df.loc[i, 'delta_SpSh_median'] = deltaSpSh_median.item()
    df.loc[i, 'delta_SpSh_mean'] = deltaSpSh_mean.item()    

    if IQR_rel_DMSOwarn.item() >1.2:                                                         #### Behnam & Conny find better threshold together
        df.loc[i, 'Tm_IQR_DMSO_Warning'] = 'Warning: high IQR for DMSO controls!'


100%|█████████████████████████████████| 176000/176000 [01:31<00:00, 1923.16it/s]


In [16]:
df

,ID,Cxsmiles,Formula,MW,Purity,PO,Salt smiles,Salt_name,Salt_ratio,MW_salt,...,Thermal_Shift,Spectral_Shift,delta_Tm_median,delta_Tm_mean,Tm_IQR_DMSO_Warning,Tm_abs_DMSO_Warning,delta_SpSh_median,delta_SpSh_mean,SpSh_IQR_DMSO_Warning,SpSh_abs_DMSO_Warning
0,BH4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.2134,2.656585,-0.2706,-0.281863,None,None,-0.246879,-0.253984,None,None
1,BH4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.3938,2.636959,-0.0902,-0.101463,None,None,-0.266506,-0.27361,None,None
2,CSCS00104537142,CC(C(=O)N1CCC(C1)(C(=O)O)C(F)(F)F)N2C=NC=N2,C11H13F3N4O3,306.2411,100.0,ini2810,NaN,NaN,NaN,0.0000,...,49.4840,2.963650,0.0,-0.011263,None,None,0.060185,0.053081,None,None
3,CSCS00133135693,CC1=NOC(=N1)C2CCN(CC2)C(=O)NC3=CC=C(F)C(F)=C3F,C15H15F3N4O2,340.3004,92.5,ini2810,NaN,NaN,NaN,0.0000,...,49.4840,2.951655,0.0,-0.011263,None,None,0.04819,0.041086,None,None
4,CSCS00093835758,CN(CC(=O)NC1CCS(=O)(=O)C1)C=2N=CN=C3C=CSC23,C13H16N4O3S2,340.4211,100.0,ini2810,NaN,NaN,NaN,0.0000,...,49.6643,2.948427,0.1803,0.169037,None,None,0.044963,0.037858,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175995,CSCS00133030229,CC(C)(C=1N=NN(CCN2C=C(Cl)C=N2)N1)S(C)(=O)=O,C10H15ClN6O2S,318.7831,100.0,ini2810,NaN,NaN,NaN,0.0000,...,NaN,NaN,None,None,None,None,None,None,None,None
175996,CSCS00027831295,CC1CN(CC(O1)C2CC2)C3=NC=NN3C,C11H18N4O,222.2868,99.0,ini2810,NaN,NaN,NaN,0.0000,...,NaN,NaN,None,None,None,None,None,None,None,None
175997,CSCS00133127202,CC=1N=C(SC1C(=O)O)C(C)NC(=O)C=2C=CC=CC2C3CCC3,C18H20N2O3S,344.4280,100.0,ini2810,NaN,NaN,NaN,0.0000,...,NaN,NaN,None,None,None,None,None,None,None,None
175998,CSCS00025925683,CC(NC(=O)NCC(C)(C)N1CCOCC1)C2=NN=CN2C,C14H26N6O2,310.3952,91.0,ini2810,NaN,NaN,NaN,0.0000,...,NaN,NaN,None,None,None,None,None,None,None,None
